In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window


In [0]:
encounters = spark.table("ehr_pipeline.bronze.synthea_encounters")
encounters.printSchema()
display(encounters.limit(5))

1.  Lowercase                  ✓
2.  START → timestamp          ✓ + extract year/month (For partitioning)
3.  STOP → timestamp           ✓
4.  duration_minutes derived   ✓ using unix_timestamp (calc durations as seconds and / 60 to get minutes)
5.  COST → float               ✓
6.  PATIENT → patient_id       ✓
7.  PROVIDER → provider_id     ✓
8.  ENCOUNTERCLASS keep as-is  ✓
9.  REASONCODE keep null       ✓
10. Deduplicate by id          ✓
11. _silver_processed_at       ✓

In [0]:
for column in encounters.columns:
  encounters = encounters.withColumnRenamed(column, column.lower())
  

In [0]:
encounters = encounters.withColumn("start", F.to_timestamp(F.col('start'), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
encounters = encounters.withColumn("stop", F.to_timestamp(F.col('stop'), "yyyy-MM-dd'T'HH:mm:ss'Z'"))

In [0]:
encounters = encounters.withColumn(
    "duration_minutes",
    ((F.unix_timestamp("stop") - F.unix_timestamp("start")) / 60).cast("integer")
)

In [0]:
encounters = encounters.withColumn('cost', F.col('cost').cast('double'))

In [0]:
encounters = encounters \
        .withColumnRenamed('patient', 'patient_id') \
        .withColumnRenamed('provider', 'provider_id')

encounters = encounters.withColumn("encounter_year", F.year("start"))
encounters = encounters.withColumn("encounter_month", F.month("start"))
encounters = encounters.withColumn("_silver_processed_at", F.current_timestamp())


withColumn("_row_rank") → temporary helper, add it
filter(_row_rank == 1)  → use it to keep latest only
drop("_row_rank")       → remove it, no longer needed

In [0]:
window = Window.partitionBy("id").orderBy(F.col("_ingested_at").desc())

encounters = encounters.withColumn("_row_rank", F.row_number().over(window)) \
                       .filter(F.col("_row_rank") == 1) \
                       .drop("_row_rank")

In [0]:
display(encounters.limit(5))

In [0]:
encounters.printSchema()

In [0]:
#counts check after dedup
display(encounters.count())